# Momentum Strategy — Critical Tier Visualizations
12-1 Cross-Sectional Momentum, S&P 500, 2000–2024

Reads the three processed parquet files from the pipeline:
- `momentum_strategy_returns.parquet` (decile returns, Long_Leg, Short_Leg, Long_Short_Spread)
- `backtest_master_panel.parquet` (stock-level panel, used to rebuild the equal-weighted market proxy)
- `backtest_panel_with_signals.parquet` (used in later notebooks for signal-level diagnostics)

Charts built here:
1. Equity Curve (Cumulative Returns)
2. Momentum Crash Event Studies (2008–09, 2020)
3. Rolling 36-Month CAPM Beta
4. Decile Return Monotonicity (with dispersion)
5. Rolling Spread Drawdowns
6. CAPM Regression Scatter & Fit Line


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

RF_MONTHLY = 0.003  # matches 04_evaluate_performance.py
ANN_FACTOR = 12

processed_dir = '../data/processed' if os.path.exists('../data/processed') else '.'

PORTFOLIO_PATH = os.path.join(processed_dir, "momentum_strategy_returns.parquet")
MASTER_PANEL_PATH = os.path.join(processed_dir, "backtest_master_panel.parquet")
SIGNALS_PATH = os.path.join(processed_dir, "backtest_panel_with_signals.parquet")  # not used in this notebook, kept for consistency

## 1. Load Data & Rebuild the Market Proxy

This mirrors the exact logic in `04_evaluate_performance.py` (Step 5) so the market series used here matches the one behind the reported 10.20% annualized market return. Do not recompute the market proxy a different way — it must tie back to the report numbers.

In [ ]:
strat_df = pd.read_parquet(PORTFOLIO_PATH)
panel_df = pd.read_parquet(MASTER_PANEL_PATH)

strat_df['rebalance_date'] = pd.to_datetime(strat_df['rebalance_date'])
panel_df['rebalance_date'] = pd.to_datetime(panel_df['rebalance_date'])

# Rebuild equal-weighted market proxy (same as 04_evaluate_performance.py)
panel_df = panel_df.sort_values(['permno', 'rebalance_date'])
panel_df['next_date'] = panel_df.groupby('permno')['rebalance_date'].shift(-1)
panel_df['holding_return_t1'] = panel_df.groupby('permno')['total_return'].shift(-1)

invalid_gap = (panel_df['next_date'] - panel_df['rebalance_date']).dt.days > 35
panel_df.loc[invalid_gap, 'holding_return_t1'] = np.nan

market_df = panel_df.groupby('rebalance_date')['holding_return_t1'].mean().reset_index()
market_df.rename(columns={'holding_return_t1': 'Market_Return'}, inplace=True)

df = pd.merge(strat_df, market_df, on='rebalance_date').dropna(subset=['Market_Return'])
df = df.sort_values('rebalance_date').reset_index(drop=True)

df['Realized_Spread'] = df['Long_Leg'] - df['Short_Leg']
df['Strategy_Excess'] = df['Long_Short_Spread']
df['Market_Excess'] = df['Market_Return'] - RF_MONTHLY

print(f"Merged panel: {len(df)} months, {df['rebalance_date'].min().date()} to {df['rebalance_date'].max().date()}")
df.head()

## 2. Equity Curve (Cumulative Returns)

Compounds `Long_Leg`, `Short_Leg`, `Long_Short_Spread`, and `Market_Return` from a base of 1.0 (i.e. \$1 invested). The `Long_Short_Spread` is a zero-cost, dollar-neutral overlay — its cumulative series represents growth of the spread P&L, not a fully-funded position.

In [ ]:
cum = pd.DataFrame({'rebalance_date': df['rebalance_date']})
for col in ['Long_Leg', 'Short_Leg', 'Long_Short_Spread', 'Market_Return']:
    cum[col] = (1 + df[col]).cumprod()

fig, ax = plt.subplots(figsize=(12, 6))

series_style = {
    'Market_Return':      dict(label='Market (Equal-Weighted)', color='#1f77b4', lw=2.2),
    'Long_Leg':            dict(label='Long Leg (Decile 10)',    color='#2ca02c', lw=1.6),
    'Short_Leg':           dict(label='Short Leg (Decile 1)',    color='#d62728', lw=1.6),
    'Long_Short_Spread':   dict(label='Long-Short Spread',       color='#7f7f7f', lw=2.2, ls='--'),
}

for col, style in series_style.items():
    ax.plot(cum['rebalance_date'], cum[col], **style)

ax.axhline(1.0, color='black', lw=0.8, alpha=0.5)
ax.set_yscale('log')
ax.set_ylabel('Growth of $1 (log scale)')
ax.set_xlabel('Date')
ax.set_title('Cumulative Returns: Momentum Strategy Legs vs. Market Proxy\n2000–2024, Monthly Rebalanced')
ax.legend(loc='upper left', frameon=False)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig('equity_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Momentum Crash Event Studies (2008–09, 2020)

Zooms into the two known momentum-crash windows. Plots `Short_Leg`, `Market_Return`, and `Long_Short_Spread` together so the short-squeeze dynamic driving spread losses is visible directly, not inferred.

In [ ]:
event_windows = {
    '2008–09 Crisis Recovery': ('2008-09-01', '2009-12-31'),
    '2020 COVID Crash & Rebound': ('2020-01-01', '2020-12-31'),
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=False)

for ax, (title, (start, end)) in zip(axes, event_windows.items()):
    window = df[(df['rebalance_date'] >= start) & (df['rebalance_date'] <= end)].copy()
    if window.empty:
        ax.set_title(f"{title}\n(no data in window)")
        continue

    # cumulative growth rebased to 1.0 at the start of the window
    for col, style in [
        ('Short_Leg', dict(label='Short Leg (Decile 1)', color='#d62728', lw=2)),
        ('Market_Return', dict(label='Market', color='#1f77b4', lw=2)),
        ('Long_Short_Spread', dict(label='Long-Short Spread', color='#7f7f7f', lw=2, ls='--')),
    ]:
        rebased = (1 + window[col]).cumprod()
        ax.plot(window['rebalance_date'], rebased, **style)

    ax.axhline(1.0, color='black', lw=0.7, alpha=0.5)
    ax.set_title(title)
    ax.set_xlabel('Date')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.tick_params(axis='x', rotation=45)
    ax.legend(loc='best', frameon=False, fontsize=9)

axes[0].set_ylabel('Growth of $1 (rebased to window start)')
fig.suptitle('Momentum Crash Event Studies: Short-Leg Squeeze vs. Market Rebound', y=1.03, fontsize=13)
plt.tight_layout()
plt.savefig('crash_event_studies.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Rolling 36-Month CAPM Beta

Rolling regression of `Long_Short_Spread` on `Market_Excess` over a 36-month trailing window (monthly step). A horizontal line marks the full-sample beta (−0.74) for comparison, to see whether the negative beta is stable or driven by a subset of the sample.

In [ ]:
ROLL_WINDOW = 36

# Full-sample beta for reference
X_full = sm.add_constant(df['Market_Excess'])
full_model = sm.OLS(df['Strategy_Excess'], X_full).fit()
full_sample_beta = full_model.params['Market_Excess']

rolling_beta = []
rolling_dates = []

for i in range(ROLL_WINDOW, len(df) + 1):
    window = df.iloc[i - ROLL_WINDOW:i]
    X = sm.add_constant(window['Market_Excess'])
    y = window['Strategy_Excess']
    model = sm.OLS(y, X).fit()
    rolling_beta.append(model.params['Market_Excess'])
    rolling_dates.append(window['rebalance_date'].iloc[-1])

beta_df = pd.DataFrame({'rebalance_date': rolling_dates, 'rolling_beta': rolling_beta})

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.plot(beta_df['rebalance_date'], beta_df['rolling_beta'], color='#9467bd', lw=1.8, label='Rolling 36M Beta')
ax.axhline(0, color='black', lw=0.8, alpha=0.6)
ax.axhline(full_sample_beta, color='#d62728', lw=1.4, ls='--',
           label=f'Full-Sample Beta ({full_sample_beta:.2f})')
ax.fill_between(beta_df['rebalance_date'], beta_df['rolling_beta'], 0,
                 where=(beta_df['rolling_beta'] < 0), color='#d62728', alpha=0.08)
ax.fill_between(beta_df['rebalance_date'], beta_df['rolling_beta'], 0,
                 where=(beta_df['rolling_beta'] >= 0), color='#2ca02c', alpha=0.08)

ax.set_ylabel('Rolling Beta (Spread vs. Market)')
ax.set_xlabel('Date')
ax.set_title('Rolling 36-Month CAPM Beta: Long-Short Spread vs. Market')
ax.legend(loc='best', frameon=False)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig('rolling_beta.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Decile Return Monotonicity (with Dispersion)

Annualized mean return per decile (`mean * 12`) with error bars showing annualized volatility (`std * sqrt(12)`), to visualize both the monotonicity of the sort and the noise around each estimate.

In [ ]:
decile_cols = [f'Decile_{i}' for i in range(1, 11) if f'Decile_{i}' in strat_df.columns]

decile_ann_ret = strat_df[decile_cols].mean() * ANN_FACTOR
decile_ann_vol = strat_df[decile_cols].std() * np.sqrt(ANN_FACTOR)

fig, ax = plt.subplots(figsize=(11, 6))
x_pos = np.arange(len(decile_cols))
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(decile_cols)))

bars = ax.bar(x_pos, decile_ann_ret.values * 100, yerr=decile_ann_vol.values * 100,
               color=colors, edgecolor='black', linewidth=0.6,
               error_kw=dict(ecolor='black', elinewidth=1, capsize=4, alpha=0.6))

ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels([c.replace('Decile_', 'D') for c in decile_cols])
ax.set_xlabel('Momentum Decile (D1 = Lowest Past Momentum, D10 = Highest)')
ax.set_ylabel('Annualized Return (%)  ± 1 SD (Annualized Vol)')
ax.set_title('Decile Return Monotonicity: Annualized Return ± Volatility by Momentum Decile')
plt.tight_layout()
plt.savefig('decile_monotonicity.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Rolling Spread Drawdowns

Builds a cumulative wealth index from `Realized_Spread` (Long_Leg − Short_Leg), computes the running maximum, and plots drawdown (`cum/cum.cummax() - 1`) as a filled area from the zero line.

In [ ]:
spread_cum = (1 + df['Realized_Spread']).cumprod()
running_max = spread_cum.cummax()
drawdown = spread_cum / running_max - 1

max_dd = drawdown.min()
max_dd_date = df['rebalance_date'].iloc[drawdown.values.argmin()]

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(df['rebalance_date'], drawdown.values * 100, 0,
                 color='#d62728', alpha=0.35)
ax.plot(df['rebalance_date'], drawdown.values * 100, color='#d62728', lw=1)
ax.axhline(0, color='black', lw=0.8)
ax.annotate(f'Max Drawdown: {max_dd:.1%}\n({max_dd_date.strftime("%b %Y")})',
            xy=(max_dd_date, max_dd * 100), xytext=(0.02, 0.08), textcoords='axes fraction',
            fontsize=9, color='#7f0000',
            arrowprops=dict(arrowstyle='->', color='#7f0000', lw=1))

ax.set_ylabel('Drawdown (%)')
ax.set_xlabel('Date')
ax.set_title('Rolling Drawdowns: Long-Short Momentum Spread (Peak-to-Trough)')
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig('rolling_drawdowns.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Maximum drawdown of the Long-Short Spread: {max_dd:.2%} (occurred {max_dd_date.date()})")

## 7. CAPM Regression Scatter & Fit Line

Scatter of `Market_Excess` (x) vs `Strategy_Excess` (y) with the OLS fit line overlaid. Annotates the fitted alpha, beta, t-statistic, and p-value directly on the chart — this is the single image that carries the report's central thesis.

In [ ]:
X = sm.add_constant(df['Market_Excess'])
y = df['Strategy_Excess']
model = sm.OLS(y, X).fit()

alpha_monthly = model.params['const']
alpha_annualized = alpha_monthly * ANN_FACTOR
beta = model.params['Market_Excess']
t_stat_alpha = model.tvalues['const']
p_value_alpha = model.pvalues['const']

x_range = np.linspace(df['Market_Excess'].min(), df['Market_Excess'].max(), 100)
y_fit = alpha_monthly + beta * x_range

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(df['Market_Excess'] * 100, df['Strategy_Excess'] * 100,
           alpha=0.5, s=35, color='#1f77b4', edgecolor='white', linewidth=0.4,
           label='Monthly Observations')
ax.plot(x_range * 100, y_fit * 100, color='#d62728', lw=2.2,
        label=f'OLS Fit: α = {alpha_annualized:.2%} (ann.), β = {beta:.2f}')

ax.axhline(0, color='black', lw=0.6, alpha=0.5)
ax.axvline(0, color='black', lw=0.6, alpha=0.5)

ax.set_xlabel('Market Excess Return (%)')
ax.set_ylabel('Strategy Excess Return (%)')
ax.set_title('CAPM Regression: Long-Short Spread Excess Return vs. Market Excess Return')

stats_text = (
    f"α (annualized): {alpha_annualized:+.2%}\n"
    f"β: {beta:.2f}\n"
    f"t-stat (α): {t_stat_alpha:.2f}\n"
    f"p-value (α): {p_value_alpha:.3f}\n"
    f"{'Significant at 5%' if p_value_alpha < 0.05 else 'Not significant at 5%'}"
)
ax.text(0.03, 0.97, stats_text, transform=ax.transAxes, fontsize=9.5,
        verticalalignment='top', horizontalalignment='left',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.9))

ax.legend(loc='lower right', frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig('capm_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

All six critical-tier charts have been saved as PNG files in the working directory (`equity_curve.png`, `crash_event_studies.png`, `rolling_beta.png`, `decile_monotonicity.png`, `rolling_drawdowns.png`, `capm_scatter.png`) for direct embedding into the Word/PDF report.